# 05 RNN/LSTM 序列建模

## 本节目标

- 理解序列数据和普通表格数据的区别。
- 学会用滑动窗口构造监督学习样本。
- 使用 LSTM 做下一步数值预测。
- 通过 loss 曲线和预测曲线观察序列模型表现。

## 背景问题

本节关注的问题是：当当前值和过去一段时间有关时，模型如何利用历史信息进行预测？

LSTM 会按时间步读取输入，并用隐状态保留已经读过的信息。这个概念可以从代码角度理解为：输入不再只是 `[batch, features]`，而是 `[batch, seq_len, features]`。

## 核心概念

- 时间步：序列中的一个位置。
- 滑动窗口：把连续片段作为输入，把后一个值作为目标。
- 隐状态：模型对历史信息的压缩表示。
- `batch_first=True`：让输入形状使用 `[batch, seq_len, features]`。

## 最小代码示例

下面先构造一条带噪声的正弦序列。这个实验用于观察 LSTM 能否根据过去窗口预测下一个值。

In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

steps = np.linspace(0, 16 * np.pi, 420)
series = np.sin(steps) + 0.08 * np.random.randn(len(steps))
series = series.astype(np.float32)

plt.figure(figsize=(8, 3))
plt.plot(series)
plt.title("Synthetic sequence")
plt.xlabel("time")
plt.ylabel("value")
plt.grid(alpha=0.3)
plt.show()

## 完整实验

这里的关键不是记住 LSTM 的门控公式，而是理解序列样本如何构造、输入 shape 如何传入模型，以及最后一个时间步的输出如何用于预测。

### 窗口构造与训练

下面用滑动窗口生成监督学习样本，再训练一个小型 LSTM。调试时可以优先检查 `x_train.shape`、`y_train.shape` 和单个 batch 的预测 shape。

In [ ]:
def make_windows(values, window_size=20):
    xs, ys = [], []
    for i in range(len(values) - window_size):
        xs.append(values[i:i + window_size])
        ys.append(values[i + window_size])
    x = torch.tensor(np.array(xs)).unsqueeze(-1)
    y = torch.tensor(np.array(ys)).unsqueeze(-1)
    return x, y

window_size = 20
x_all, y_all = make_windows(series, window_size)
split = int(len(x_all) * 0.8)
x_train, y_train = x_all[:split], y_all[:split]
x_test, y_test = x_all[split:], y_all[split:]

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)

In [ ]:
class LSTMRegressor(nn.Module):
    def __init__(self, hidden_dim=32):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_dim, batch_first=True)
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        sequence_output, _ = self.lstm(x)
        last_step = sequence_output[:, -1, :]
        return self.output(last_step)

model = LSTMRegressor(hidden_dim=32)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
losses = []

for epoch in range(80):
    model.train()
    pred = model(x_train)
    loss = criterion(pred, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

model.eval()
with torch.no_grad():
    test_pred = model(x_test)
    test_loss = criterion(test_pred, y_test).item()

print(f"final train loss={losses[-1]:.4f}, test loss={test_loss:.4f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.title("LSTM training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 3))
plt.plot(y_test.numpy()[:120], label="target")
plt.plot(test_pred.numpy()[:120], label="prediction")
plt.title("One-step prediction")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 实验观察

从运行结果可以看到，loss 下降后，预测曲线能跟随正弦序列的大致趋势。由于序列中有噪声，模型不会完全贴合每个点，这反而是合理现象。窗口长度和隐藏维度都会影响模型能利用多少历史信息。

## 常见错误

- 输入缺少 feature 维度，导致 LSTM shape 不匹配。
- 忘记设置 `batch_first=True`，但仍按 `[batch, seq, feature]` 传入。
- 构造窗口时目标错位，导致模型学到错误对应关系。
- 学习率过大，loss 震荡明显。

初学者容易在这里混淆“序列长度”和“特征数”。一个时间步可以有多个特征，但本实验每个时间步只有一个数值。

In [ ]:
print("one sample input shape:", x_train[:1].shape)
print("one prediction shape:", model(x_train[:1]).shape)

## 概念问答

**Q：为什么要用滑动窗口？**  
A：它把连续序列转换成监督学习样本，让模型可以根据一段历史预测下一个值。

**Q：LSTM 一定比 MLP 好吗？**  
A：不一定。LSTM 更适合利用顺序关系，但是否更好取决于数据、任务和训练设置。

**Q：为什么只取最后一个时间步？**  
A：这个实验是用整个窗口预测下一个值，最后一个时间步的输出可以看作对窗口信息的汇总。

## 小结

序列建模的关键在于样本构造和形状管理。先跑通窗口预测，再扩展到更复杂的文本或时间序列任务，会更稳。